# Colab Base para el Trabajo Práctico (versión 6)
Dada la diferencia que existe entre los dataset en la **geolocalización, operación, tipo de propiedad, moneda**, este filtro se basara en tales campos.

Imputación de lat y lon utilizando location 2 y 3

In [ ]:
import pandas as pd
import sqlite3

import sklearn as sk
from sklearn import model_selection
from sklearn import ensemble
from sklearn import metrics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 0. Lectura de datos

In [ ]:
# TODO: Cambiar para que apunte al directorio correcto
DIR = "/content/drive/MyDrive/UBA/Especializacion/DM/Data"

In [ ]:
engine = sqlite3.connect(f"{DIR}/entrenamiento.db")

df_ent = pd.read_sql("SELECT * FROM entrenamiento", engine, index_col="id")
# df_ent.head(3)

In [ ]:
# cantidad de filas y columnas
df_ent.shape

(1292674, 17)

In [ ]:
# lista de columnas del dataframe
df_ent.columns

Index(['description', 'address', 'lat', 'lon', 'publication_date',
       'publisher_id', 'features', 'location_0', 'location_1', 'location_2',
       'location_3', 'location_4', 'operation_type', 'property_type', 'source',
       'price', 'currency_type'],
      dtype='object')

In [ ]:
# Dataset a predecir:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")

## 1. Entender los datos (AID)

In [ ]:


print(f"Dataset a predecir - Operaciones:\n-{df_ap.operation_type.unique()}\n")
print(f"Dataset a entrenar - Operaciones:\n-{df_ent.operation_type.unique()}")
print("-"*90)

print(f"Dataset a predecir - Propiedad:\n-{df_ap.property_type.unique()}\n")
print(f"Dataset a entrenar - Propiedad:\n-{df_ent.property_type.unique()}")
print("-"*90)

print(f"Dataset a predecir - País:\n-{df_ap.location_0.unique()}\n")
print(f"Dataset a entrenar - País:\n-{df_ent.location_0.unique()}\n")
print("-"*90)
print(f"Dataset a predecir - Provincia:\n-{df_ap.location_1.unique()}\n")
print(f"Dataset a entrenar - Provincia:\n-{df_ent.location_1.unique()}\n")
print("-"*90)
print(f"Dataset a predecir - Ciudad:\n-{df_ap.location_2.unique()}\n")
print(f"Dataset a entrenar - Ciudad:\n-{df_ent.location_2.unique()}")
print("-"*90)

print(f"Dataset a predecir - Moneda:\n-{df_ap.currency_type.unique()}\n")
print(f"Dataset a entrenar - Moneda:\n-{df_ent.currency_type.unique()}\n")
print("-"*90)
print(f"Dataset a predecir - Fuente de datos:\n-{df_ap.source.unique()}\n")
print(f"Dataset a entrenar - Fuente de datos:\n-{df_ent.source.unique()}\n")
print("-"*90)

Dataset a predecir - Operaciones:
-['venta']

Dataset a entrenar - Operaciones:
-['venta' None 'alquiler' 'temporal' 'alquiler / temporal'
 'venta / temporal' 'venta / alquiler' 'venta / alquiler / temporal'
 'traspaso' 'renta' 'alquiler temporal' 'sin operacion']
------------------------------------------------------------------------------------------
Dataset a predecir - Propiedad:
-['casa' 'departamento' 'cochera']

Dataset a entrenar - Propiedad:
-['galpón' None 'casa' 'terreno' 'departamento' 'local comercial' 'oficina'
 'cochera' 'villa' 'terrenos' 'garage' 'oficina comercial' 'ph'
 'consultorio' 'bodega-galpón' 'hotel' 'edificio' 'depósito' 'campo'
 'quinta vacacional' 'fondo de comercio' 'desarrollo vertical'
 'desarrollo horizontal' 'bóveda, nicho o parcela' 'cama náutica'
 'casa-duplex' 'galpon' 'quinta' 'cabana' 'penthouse' 'loft'
 'fondo-de-comercio' 'inmueble-productivo' 'local' 'fabrica' 'nave'
 'deposito' 'oficinas' 'departamentos' 'casas' 'comercios' 'depósitos'
 'empr

In [ ]:
df_ent.currency_type.value_counts()

,count
currency_type,
dolares,112635
pesos,196


In [ ]:
df_ap.currency_type.value_counts()

,count
currency_type,
dolares,13463
pesos,8


In [3]:
df_ent.operation_type.value_counts()

NameError: name 'df_ent' is not defined

## 2. Limpiar y transformar los datos (DM)

In [ ]:
# Guardo el dataframe en un csv, para luego acceder más facil.
df_ent.to_csv("/content/drive/MyDrive/UBA/Especializacion/DM/Data/raw/entrenamiento.csv")

In [ ]:
# Entrenamiento se basa en los parámentreos lon y lat, este filtro lo haremos por geolocalización,
#  operación, tipo de propiedad, moneda, fuente de datos
filtro = ((df_ent["location_0"] == 'Argentina') &
 (df_ent["location_1"].isin(['Ciudad Autónoma de Buenos Aires','Capital Federal'])) &
          (df_ent["operation_type"] == "venta") &
          (df_ent["property_type"].isin(['casa','departamento', "cochera"])) &
          (df_ent["currency_type"].isin(['dolares']))   &
          (df_ent["price" ] < 700000))
df_ent = df_ent.loc[filtro]
df_ent.shape
# test = df_ent.loc[filtro]
# test.shape

(108194, 17)

In [ ]:
df_ent.info()

<class 'pandas.core.frame.DataFrame'>
Index: 112831 entries, 3659 to 1270469
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   description       112831 non-null  object 
 1   address           112536 non-null  object 
 2   lat               112831 non-null  float64
 3   lon               112831 non-null  float64
 4   publication_date  45189 non-null   object 
 5   publisher_id      83139 non-null   object 
 6   features          112716 non-null  object 
 7   location_0        112831 non-null  object 
 8   location_1        112831 non-null  object 
 9   location_2        112827 non-null  object 
 10  location_3        99351 non-null   object 
 11  location_4        7849 non-null    object 
 12  operation_type    112831 non-null  object 
 13  property_type     112831 non-null  object 
 14  source            111400 non-null  object 
 15  price             112771 non-null  float64
 16  currency_type     112

In [ ]:
# Imputación de valores perdidos
# Promedio del precio
promedio_casa = df_ent.loc[df_ent["property_type"] == "casa",  "price" ].mean()
promedio_departamento = df_ent.loc[df_ent["property_type"] == "departamento",  "price" ].mean()

df_ent.loc[(df_ent["price"].isna()) & (df_ent["property_type"] == "casa"), "price" ] = promedio_casa
df_ent.loc[(df_ent["price"].isna()) & (df_ent["property_type"] == "departamento"), "price" ] = promedio_departamento

In [ ]:
# La creación de modelos requiere que no haya valores perdidos
# por eso llenamos todo con 0 a lo bestia
# TODO: mejorar la imputación de valores perdidos
df_ent = df_ent.fillna(0)

## 3. Entrenamiento del modelos (AA) - ⛔⛔⛔ NO TOCAR ⛔⛔⛔

In [ ]:
# La creación de modelos requiere que todo el dataframe sea numérico
# Me quedo con las columnas numéricas solamente
# TODO: traducir las columnas con datos no numéricos a numéricos para que mejoren los modelos
df_ent = df_ent.select_dtypes('number')

X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

In [ ]:
X.head()

,lat,lon
id,,
3659,-34.558289,-58.443645
70676,-34.563152,-58.494072
70677,-34.603661,-58.388000
70678,-34.607761,-58.386929
70679,-34.617531,-58.376518


In [ ]:
X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

# Definimos el valor de los hiperparámetros a usar por el modelo
n_estimators = 50
max_depth = 5

### NO CAMBIAR RandomForestRegressor por otro modelo
reg = sk.ensemble.RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, n_jobs=-1, random_state=42)

# Entrenamos el modelo
_ = reg.fit(X_train, y_train)

# Cálculo del error en entrenamiento (train)
y_pred = reg.predict(X_train)
score_train = sk.metrics.root_mean_squared_error(y_train, y_pred)

# Cálculo del error en prueba (test)
y_pred = reg.predict(X_test)
score_test  = sk.metrics.root_mean_squared_error(y_test,  y_pred)

print(f"{n_estimators=} -- {max_depth=} --> {score_train=:.2f} - {score_test=:.2f}")

n_estimators=50 -- max_depth=5 --> score_train=121421.87 - score_test=120569.05


## 4. Solución para subir Kaggle

In [ ]:
df_ap = pd.read_csv(f"{DIR}/a_predecir.csv", index_col="id")
# df_ap.head(2)

In [ ]:
X = df_ent[df_ent.columns.drop('price')]
y = df_ent['price']

# Entrenamos el modelo con todos los datos de entrenamiento.csv
reg.fit(X, y)

RandomForestRegressor(max_depth=5, n_estimators=50, n_jobs=-1, random_state=42)

In [ ]:
df_ap = df_ap.fillna(0)

df_ap = df_ap.select_dtypes('number')

X_ap = df_ap[X.columns]

# Predecimos los precios del dataset a predecir
y_pred_ap = reg.predict(X_ap)
y_pred_ap

array([107721.17472484, 107721.17472484, 107721.17472484, ...,
       166787.20474835, 107721.17472484, 182087.50152256])

In [ ]:
# Lleno el precio de df_ap con las predicciones
df_ap["price"] = y_pred_ap

# Grabo el df_ap en un archivo csv para subir a Kaggle
df_ap["price"].to_csv("/content/drive/MyDrive/UBA/Especializacion/DM/Primera-entrega/solucion-version6-3.csv")